# Step 2 — Extract O'Byrne (2015) Appendix A and Pre-label

## Purpose
Extract the 311 adoption table from O'Byrne (2015) and assign initial status labels.
This is the **only systematic academic source** of binary adopter/non-adopter status
for US cities, covering all cities with population > 100,000 in 2003.

## What O'Byrne contains
Appendix A: one row per US city with 2003 population > 100,000. Columns:
- `Adopt` (1 = adopted 311 by 2012, 0 = did not adopt, NA = excluded from regression)
- `Mayor` (mayor-council government indicator)
- `Population` (2003 population)
- `Violent Crime` (violent crime rate per 100k)
- `Damage` (property damage)

**O'Byrne does not record adoption year — only binary status as of 2012.**

## Pre-labeling rule
| adopt_by2012 | pre_label |
|---|---|
| 1 | `adopted_by2012` — confirmed adopter, year unknown |
| 0 | `not_adopted_by2012` — confirmed non-adopter as of 2012 |
| NA | `unclear` — in O'Byrne sample but excluded from his regression |

## Source
O'Byrne, J.C. (2015). *The Diffusion and Evolution of 311 Citizen Service Centers in
American Cities from 1996 to 2012.* Virginia Tech PhD Dissertation.
https://vtechworks.lib.vt.edu/handle/10919/52634

## Requirement
`pdftotext` from poppler: `brew install poppler` (macOS)

## Output
`data_clean/obyrne_extracted.csv`

In [1]:
import re
import subprocess
import pandas as pd

RAW   = '../data_raw/'
CLEAN = '../data_clean/'
PDF_PATH = RAW + 'obyrne_2015_dissertation.pdf'

## 1. Extract PDF text with layout preservation

In [2]:
result = subprocess.run(
    ['pdftotext', '-layout', PDF_PATH, '-'],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f'pdftotext failed: {result.stderr}')

full_text = result.stdout
print(f'Extracted {len(full_text):,} characters from PDF')

Extracted 444,749 characters from PDF


## 2. Isolate Appendix A

The PDF has multiple occurrences of 'Appendix A.' (two in the TOC, one for the actual
content). We take the last occurrence and stop at 'Appendix B.'.

In [3]:
matches   = [m.start() for m in re.finditer(r'Appendix A\.', full_text)]
app_start = matches[-1]
app_end_m = re.search(r'Appendix B\.', full_text[app_start:])
app_end   = app_start + (app_end_m.start() if app_end_m else 40_000)
app_text  = full_text[app_start:app_end]

print(f'Appendix A isolated: {len(app_text):,} characters')
print('--- First 400 chars ---')
print(app_text[:400])

Appendix A isolated: 19,982 characters
--- First 400 chars ---
Appendix A. Logistic Regression Model Data, All Cities and Consolidated
Governments over 100,000 (2003)

                              Adopt   Mayor                  Violent
Governments                   (1=Y)   (1=Y)     Population   Crime     Damage
New York, NY (includes 5
counties)                       1           1   8,098,066     734.1        60
Los Angeles, CA                 1           1


## 3. Parse rows — handle multi-line city names

With `-layout`, each data row ends with five numeric columns separated by large gaps.
Some city names span two printed lines (e.g. 'New York, NY (includes 5 / counties)').

Strategy:
1. Match lines whose tail fits the 5-column numeric pattern.
2. Extract the leading city-name fragment.
3. If the fragment lacks a state code (', XX'), prepend the previous non-blank line.

In [4]:
NUM_COLS = re.compile(
    r'^(.+?)'                          # city name fragment
    r'\s{2,}'                          # large gap
    r'(\d|NA)'                         # Adopt
    r'\s+(\d|NA)'                      # Mayor
    r'\s+([\d,]+|NA)'                  # Population
    r'\s+([\d.]+|NA)'                  # Violent Crime
    r'\s+([\d,]+|NA)\s*$'             # Damage
)
STATE_RE = re.compile(r',\s*[A-Z]{2}(\s|\(|$)')
SKIP_RE  = re.compile(
    r'^(Governments|Adopt|Mayor|Population|Violent|Crime|Damage|Sources|Date|\d+\s*$)',
    re.IGNORECASE
)

lines     = app_text.split('\n')
rows      = []
prev_name = ''

for line in lines:
    m = NUM_COLS.match(line)
    if not m:
        stripped = line.strip()
        if stripped and not SKIP_RE.match(stripped):
            prev_name = stripped
        elif not stripped:
            prev_name = ''
        continue

    fragment = m.group(1).strip()
    if STATE_RE.search(fragment):
        city_raw = fragment
    elif prev_name:
        city_raw = (prev_name + ' ' + fragment).strip()
    else:
        city_raw = fragment

    def parse_num(val, as_float=False):
        if val == 'NA':
            return None
        val = val.replace(',', '')
        return float(val) if as_float else int(val)

    rows.append(dict(
        city_raw           = city_raw,
        adopt_by2012       = parse_num(m.group(2)),
        mayor              = parse_num(m.group(3)),
        pop_2003           = parse_num(m.group(4)),
        violent_crime_2003 = parse_num(m.group(5), as_float=True),
        damage_2003        = parse_num(m.group(6)),
    ))
    prev_name = ''

print(f'Parsed {len(rows)} rows')
print(f'First 5: {[r["city_raw"] for r in rows[:5]]}')

Parsed 242 rows
First 5: ['New York, NY (includes 5 counties)', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'County of, PA']


## 4. Extract state abbreviation and apply manual fixes for page-break names

In [5]:
GET_STATE = re.compile(r',\s*([A-Z]{2})(?:\s|\(|$)')

# Known page-break fragments → correct full name
FIXES = {
    'counties)':                         'New York, NY',
    'County of, PA':                     'Philadelphia, PA',
    'portion of Marion County)':         'Indianapolis, IN',
    '(consolidated with Duval County)':  'Jacksonville, FL',
    'County of, CA':                     'San Francisco, CA',
    'County, NC (City Portion)':         'Charlotte, NC',
    'of Suffolk County)':                'Boston, MA',
    'Washington, District of Columbia':  'Washington, DC',
    'CO':                                'Denver, CO',
    'TN':                                'Nashville, TN',
    '(City portion)':                    'Miami-Dade, FL',
    'of, HI (City Portion)':             'Honolulu, HI',
    'County Government, KY':             'Lexington-Fayette, KY',
    '(City portion), KY':                'Louisville, KY',
    'Government, GA':                    'Columbus, GA',
    'County Unified Government, KS':     'Kansas City, KS',
    '(Ventura), CA':                     'Ventura, CA',
    'GA (City portion)':                 'Athens-Clarke, GA',
}

df = pd.DataFrame(rows)
df['city_raw'] = df['city_raw'].apply(lambda x: FIXES.get(x.strip(), x))

df['state_abbr'] = df['city_raw'].apply(
    lambda x: (GET_STATE.findall(x) or [None])[-1]
)
df.loc[df['city_raw'] == 'Washington, DC', 'state_abbr'] = 'DC'

print(f'Total rows         : {len(df)}')
print(f'Missing state      : {df["state_abbr"].isna().sum()}')
if df['state_abbr'].isna().any():
    print(df[df['state_abbr'].isna()][['city_raw']].to_string())

Total rows         : 242
Missing state      : 0


## 5. Assign pre_label based on adopt_by2012

In [6]:
def assign_label(val):
    if val == 1:
        return 'adopted_by2012'
    elif val == 0:
        return 'not_adopted_by2012'
    else:  # NA
        return 'unclear'

df['pre_label'] = df['adopt_by2012'].apply(assign_label)
df['source']    = "O'Byrne (2015) Appendix A"

print('Pre-label distribution:')
print(df['pre_label'].value_counts().to_string())
print()
print('Adopters (adopt_by2012=1):')
adopters = df[df['pre_label'] == 'adopted_by2012'][['city_raw', 'state_abbr', 'pop_2003']]
print(adopters.sort_values('pop_2003', ascending=False).to_string(index=False))

Pre-label distribution:
pre_label
not_adopted_by2012    180
adopted_by2012         53
unclear                 9

Adopters (adopt_by2012=1):
                            city_raw state_abbr  pop_2003
  New York, NY (includes 5 counties)         NY 8098066.0
                     Los Angeles, CA         CA 3838838.0
                         Chicago, IL         IL 2898374.0
                         Houston, TX         TX 2041081.0
                    Philadelphia, PA         PA 1495903.0
                          Dallas, TX         TX 1230302.0
                     San Antonio, TX         TX 1212789.0
                         Detroit, MI         MI  927766.0
                        San Jose, CA         CA  909890.0
                   San Francisco, CA         CA  772065.0
                        Columbus, OH         OH  726151.0
                          Austin, TX         TX  682319.0
                       Charlotte, NC         NC  668003.0
                       Baltimore, MD         MD 

## 6. Save

In [7]:
col_order = [
    'city_raw', 'state_abbr', 'adopt_by2012', 'mayor',
    'pop_2003', 'violent_crime_2003', 'damage_2003',
    'pre_label', 'source'
]
df = df[col_order]

out_path = CLEAN + 'obyrne_extracted.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows → {out_path}')
print()
print('Summary:')
print(f'  Total cities in O\'Byrne sample : {len(df)}')
print(f'  adopted_by2012                 : {(df.pre_label=="adopted_by2012").sum()}')
print(f'  not_adopted_by2012             : {(df.pre_label=="not_adopted_by2012").sum()}')
print(f'  unclear                        : {(df.pre_label=="unclear").sum()}')

Saved 242 rows → ../data_clean/obyrne_extracted.csv

Summary:
  Total cities in O'Byrne sample : 242
  adopted_by2012                 : 53
  not_adopted_by2012             : 180
  unclear                        : 9
